# Real-model evaluation in Google Colab - Llama 3.2 3B Instruct

**Model under test:** `meta-llama/Llama-3.2-3B-Instruct` (revision `0cb88a4f764b7a12671c53f0838cd831a0843b95`; pin a commit sha before a study run).
**Description:** Instruction-tuned 3.2B Llama model with a different tokenizer and pretraining lineage.
**Parameters:** 3.2B | **fp16 weights:** ~6.5 GB | **Licence:** Llama 3.2 Community Licence
**Role in the study:** Adds the Llama family to the primary comparison; the only run so far pinned to a commit sha.
**Status:** gated: licence accepted and access granted; revision pinned, so no 'main' warning in the summary

**Where the results belong:** `results/colab_real_model/Llama-3.2-3B-Instruct/`; curate and summarise them with
`python3 scripts/summarise_real_model_results.py results/colab_real_model/Llama-3.2-3B-Instruct`.

**Parallel sessions:** this notebook mirrors its artifacts to
`MyDrive/apertus_runs/phase8_real_colab/Llama-3.2-3B-Instruct` and exports
`apertus_phase8_real_colab_Llama-3.2-3B-Instruct_export.zip`, so runs of other models cannot overwrite them.
This notebook runs the existing Phase 7/8 Apertus Eval Prep platform against one selected open-weight Transformers model on an interactive GPU runtime. It is an evaluation experiment, not foundation-model training or fine-tuning.

This produces **experimental real-model evidence**, not a production benchmark. Colab hardware is ephemeral and can vary by session. Results apply only to the recorded model/tokenizer revisions, prompt, dataset, decoding, and runtime. Do not commit tokens, secrets, private datasets, downloaded model caches, or large generated runs. Release-gate results are engineering policy aids and are not production approval.

In [1]:
import importlib.metadata as importlib_metadata
import json
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_PATH = "/content/apertus-eval-prep"
repo = Path(REPOSITORY_PATH).expanduser()
if not (repo / "pyproject.toml").exists():
    repository_url = os.environ.get("APERTUS_REPO_URL", "https://github.com/Shivani767/apertus-eval-prep.git")
    if not repository_url:
        raise RuntimeError("Open/clone the repository first or set APERTUS_REPO_URL privately.")
    subprocess.run(["git", "clone", "--depth", "1", repository_url, str(repo)], check=True)
os.chdir(repo)

# The package lives under src/. An editable install only registers a .pth file,
# which this already-running kernel will not re-read until it restarts, so put
# src/ on the path explicitly. pytest does the same via pythonpath in pyproject.
source_root = repo / "src"
if not (source_root / "apertus_eval_prep" / "__init__.py").exists():
    raise RuntimeError("Missing package sources at %s. Delete %s and re-run this cell." % (source_root, repo))
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

def _install(extra):
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-e", ".[%s]" % extra],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        print(result.stdout[-4000:])
        print(result.stderr[-4000:])
        raise RuntimeError("pip install -e .[%s] failed; see the pip output above." % extra)
    print("installed -e .[%s]" % extra)

_install("dev")
_install("real-model")

import apertus_eval_prep
print("apertus_eval_prep", apertus_eval_prep.__version__)
print("imported from", apertus_eval_prep.__file__)
for package in ("torch", "transformers", "accelerate", "pyyaml"):
    try:
        print(package, importlib_metadata.version(package))
    except importlib_metadata.PackageNotFoundError:
        print(package, "UNAVAILABLE")
subprocess.run([sys.executable, "-m", "apertus_eval_prep", "platform-run", "--config", "configs/platform_smoke.yaml", "--out", "runs/phase8_real_colab/mock_smoke"], check=True)
print("Offline mock smoke completed. Expected evidence mode: MOCK.")


installed -e .[dev]
installed -e .[real-model]
apertus_eval_prep 0.5.0
imported from /content/apertus-eval-prep/src/apertus_eval_prep/__init__.py
torch 2.11.0+cu128
transformers 5.16.1
accelerate 1.14.0
pyyaml 6.0.3
Offline mock smoke completed. Expected evidence mode: MOCK.


In [2]:
import json
# pyright: reportUndefinedVariable=false
from apertus_eval_prep.utils.runtime_profile import profile_runtime

# Prefer the config-cell values, but never crash if that cell was not run.
_cfg = {key: globals().get(key) for key in ('DEVICE', 'DTYPE', 'QUANTIZATION')}
DEVICE = _cfg['DEVICE'] or 'cuda'
DTYPE = _cfg['DTYPE'] or 'float16'
QUANTIZATION = _cfg['QUANTIZATION'] or 'none'
print('config inherited:', {k: v for k, v in _cfg.items() if v is not None} or 'none (defaults used)')

runtime_profile = profile_runtime(device=DEVICE, precision=DTYPE, quantization=QUANTIZATION)

gpu_name = runtime_profile.get('gpu_name')
gpu_count = runtime_profile.get('gpu_count') or 0
torch_version = str(runtime_profile.get('torch_version') or '')
gpu_memory = runtime_profile.get('gpu_memory')
gpu_unit = runtime_profile.get('gpu_memory_unit') or ''
status = 'GPU READY' if (gpu_name and gpu_count) else 'NO GPU'

print('[%s] device=%s dtype=%s quantization=%s' % (status, DEVICE, DTYPE, QUANTIZATION))
print('  torch      :', torch_version or 'not installed')
print('  cuda       :', runtime_profile.get('cuda_version'))
print('  gpu        :', '%s x%s' % (gpu_name or 'none', gpu_count))
print('  gpu memory :', ('%s %s' % (gpu_memory, gpu_unit)).strip() or 'unknown')
print('  cpu count  :', runtime_profile.get('cpu_count'))
print('  environment:', runtime_profile.get('runtime_environment'))
print('  measured   :', runtime_profile.get('hardware_measurement_status'))

if not gpu_name or not gpu_count:
    print()
    print('!! No GPU detected. The real-model cells will refuse to run.')
    print('   Fix: Runtime -> Change runtime type -> GPU (T4/L4), then')
    print('        Runtime -> Restart session, and re-run this cell.')
    if '+cpu' in torch_version:
        print('   NOTE: a CPU-only torch build is installed. After enabling a GPU,')
        print('         do NOT reinstall torch or pip will fetch another +cpu wheel.')
        print('         Use instead: pip install transformers accelerate')
elif '+cpu' in torch_version:
    print()
    print('!! GPU is present but torch is a CPU-only build (+cpu).')
    print('   Install the CUDA build, e.g.:')
    print('     pip install torch --index-url https://download.pytorch.org/whl/cu121')
    print('   then restart the session and re-run this cell.')

print()
print('Full profile:')
print(json.dumps(runtime_profile, indent=2, default=str))
print()
print('Colab hardware is ephemeral and may differ between sessions.')
print('Do not treat this profile as production serving hardware.')

config inherited: none (defaults used)
[GPU READY] device=cuda dtype=float16 quantization=none
  torch      : 2.11.0+cu128
  cuda       : 12.8
  gpu        : Tesla T4 x1
  gpu memory : 14.56 GiB
  cpu count  : 2
  environment: google_colab
  measured   : measured

Full profile:
{
  "schema_version": "1.0",
  "runtime_environment": "google_colab",
  "os": {
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
    "system": "Linux",
    "release": "6.6.122+",
    "machine": "x86_64",
    "processor": "x86_64",
    "cpu_count": 2
  },
  "python": {
    "version": "3.13.15",
    "implementation": "CPython",
    "executable": "/usr/bin/python3"
  },
  "torch_version": "2.11.0+cu128",
  "cuda_version": "12.8",
  "gpu_name": "Tesla T4",
  "gpu_count": 1,
  "gpu_memory": 14.56,
  "gpu_memory_unit": "GiB",
  "cpu_count": 2,
  "device_selection": "cuda",
  "precision": "float16",
  "quantization": "none",
  "hardware_measured": true,
  "hardware_measurement_status": "measured",
  "latency_met

## 1. Select the model and decoding settings

Edit only the variables in the next cell. Use a smaller model if GPU memory is limited. Begin with float16/no quantization. Use int8/int4 only after the baseline succeeds and optional quantization dependencies are installed. Pin a model revision if available. Do not enable `trust_remote_code` unless required and reviewed. No access token belongs in this notebook.

In [3]:

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
MODEL_REVISION = "0cb88a4f764b7a12671c53f0838cd831a0843b95"
TOKENIZER_ID = None
DEVICE = "cuda"
DTYPE = "float16"
QUANTIZATION = "none"
MAX_NEW_TOKENS = 128
TEMPERATURE = 0.0
TOP_P = 1.0
SEED = 11

In [4]:
# pyright: reportUndefinedVariable=false
import json
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path
from apertus_eval_prep.utils.runtime_profile import profile_runtime
from apertus_eval_prep.utils.serialization import read_yaml, write_yaml

OUTPUT_ROOT = Path("runs/phase8_real_colab")
CONFIG_ROOT = OUTPUT_ROOT / "configs"
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
STUDY_DIR = Path("configs/studies/phase8_sarvam_application_study")
PRIVATE_CONFIG = STUDY_DIR / "local_user_config.yaml"
TEMPLATE = STUDY_DIR / "local_user_config.example.yaml"
REVISION = None if str(MODEL_REVISION).startswith("OPTIONAL_") else MODEL_REVISION
TOKENIZER = TOKENIZER_ID or MODEL_ID
if str(MODEL_ID).startswith("YOUR_") or not str(MODEL_ID).strip():
    raise RuntimeError("Edit MODEL_ID before running the real-model cells.")
if DEVICE != "cuda":
    raise RuntimeError("This workflow is intentionally CUDA-only; it will not silently fall back to CPU.")

def require_cuda():
    profile = profile_runtime(device=DEVICE, precision=DTYPE, quantization=QUANTIZATION)
    print(json.dumps(profile, indent=2, default=str))
    if not profile.get("gpu_name") or not profile.get("gpu_count"):
        raise RuntimeError("CUDA GPU is unavailable. Enable a Colab GPU runtime before continuing.")

def _base(config):
    return config["experiment"]["base"] if "experiment" in config else config

def _set_identity(config, output_dir):
    base = _base(config)
    adapter = base.setdefault("adapter", {})
    adapter.update(kind="local_transformers", model_id=MODEL_ID, revision=REVISION)
    params = adapter.setdefault("params", {})
    params.update(tokenizer_id=TOKENIZER, tokenizer_revision=REVISION, trust_remote_code=False, device=DEVICE, dtype=DTYPE, quantization=QUANTIZATION, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE, top_p=TOP_P, do_sample=TEMPERATURE > 0.0, timeout_s=600)
    base.setdefault("runtime", {}).update(device=DEVICE, precision=DTYPE, quantization=QUANTIZATION, timeout_s=600)
    base.setdefault("decoding", {}).update(seed=SEED, temperature=TEMPERATURE, top_p=TOP_P, max_new_tokens=MAX_NEW_TOKENS)
    base.setdefault("cost", {}).update(input_per_million=None, output_per_million=None, currency="USD", source="manual_config", effective_date=None, estimate_label="DERIVED_ESTIMATE")
    base.setdefault("evidence", {}).update(mode="LOCAL_REAL_MODEL", runtime_environment="google_colab", real_model_execution=True, external_provider_execution=False, hardware_measured=True, human_reviewed=False, pricing_source="manual_config")
    base.setdefault("reporting", {})["output_dir"] = str(output_dir)
    if isinstance(config.get("model"), dict):
        config["model"].update(id=MODEL_ID, revision=REVISION, tokenizer_id=TOKENIZER)
    if "experiment" in config:
        experiment = config["experiment"]
        experiment["baseline"].update(seed=SEED, temperature=TEMPERATURE, top_p=TOP_P, prompt_template="base", model_revision=REVISION, backend="local_transformers", precision=DTYPE, quantization=QUANTIZATION)
        experiment.setdefault("factors", {}).update(model_revision=[REVISION], backend=["local_transformers"], precision=[DTYPE], quantization=[QUANTIZATION], seed=[SEED], temperature=[TEMPERATURE], top_p=[TOP_P])
        base["prompt"] = {"prompt_id": "sarvam-core", "version": "v1", "template": "base"}
    return config

def write_private_config():
    write_yaml(PRIVATE_CONFIG, _set_identity(read_yaml(TEMPLATE), OUTPUT_ROOT / "local_baseline"))
    return PRIVATE_CONFIG

def write_experiment_config(template_name, destination, output_dir, *, seeds=None, prompt_templates=None):
    config = _set_identity(read_yaml(STUDY_DIR / template_name), output_dir)
    if "experiment" in config:
        if seeds is not None:
            config["experiment"].setdefault("factors", {})["seed"] = list(seeds)
            config["experiment"]["baseline"]["seed"] = seeds[0]
        if prompt_templates is not None:
            config["experiment"].setdefault("factors", {})["prompt_template"] = list(prompt_templates)
            config["experiment"]["baseline"]["prompt_template"] = prompt_templates[0]
    write_yaml(destination, config)
    return destination

def gpu_status():
    """Report GPU memory so a memory problem is visible before a step fails."""
    import torch
    if not torch.cuda.is_available():
        return {"cuda": False}
    free, total = torch.cuda.mem_get_info()
    return {"cuda": True, "gpu": torch.cuda.get_device_name(0), "free_gib": round(free / 2**30, 2), "total_gib": round(total / 2**30, 2), "allocated_gib": round(torch.cuda.memory_allocated() / 2**30, 3), "reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 3)}

def free_gpu():
    """Release kernel-held memory before a platform subprocess loads the model.

    Every platform step runs in a fresh process, but this kernel can still hold a model
    from the manual GPU test above. If that model is not released, the subprocess runs
    out of memory even when the model itself fits on the GPU.
    """
    import gc
    for name in ("model", "tokenizer", "inputs", "generated"):
        globals().pop(name, None)
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass
    return gpu_status()

DRIVE_MIRROR = Path("/content/drive/MyDrive/apertus_runs/phase8_real_colab/Llama-3.2-3B-Instruct")

def run_platform(*args, heartbeat_s=30):
    """Run one platform CLI step, streaming a liveness heartbeat while it works.

    A platform run writes its run directory only when the whole run finishes,
    so a long suite prints nothing until it is done. The heartbeat proves the
    step is still alive and records how long each step actually took.
    """
    # Free anything this kernel still holds (manual GPU test) before the subprocess loads the model.
    print("    GPU before step:", free_gpu(), flush=True)
    command_name = str(args[0]) if args else "platform"
    command = [sys.executable, "-u", "-m", "apertus_eval_prep", *map(str, args)]
    print("$ " + " ".join(command), flush=True)
    started = time.time()
    stop = threading.Event()

    def _heartbeat():
        while not stop.wait(heartbeat_s):
            print("    ... %s still running (%.1f min elapsed)" % (command_name, (time.time() - started) / 60.0), flush=True)

    beat = threading.Thread(target=_heartbeat, daemon=True)
    beat.start()
    try:
        subprocess.run(command, check=True)
    except subprocess.CalledProcessError as exc:
        raise RuntimeError("Platform command failed. If the failure is a CUDA out-of-memory error: (1) Runtime > Restart session, then re-run this cell, so no model from the manual GPU test is still held; (2) lower MAX_NEW_TOKENS; (3) set QUANTIZATION to int8 for models above about 4B, and int4 only after the baseline succeeds and the optional quantization dependency is installed; (4) run the manual GPU test AFTER the workflow steps, or skip it. free_gpu() runs automatically before every step. No condition was silently changed.") from exc
    finally:
        stop.set()
    print("    done %s in %.1f min" % (command_name, (time.time() - started) / 60.0), flush=True)


def print_artifacts(root):
    """List every finished run directory under root with its scored-example count."""
    finished = run_dirs(root)
    if not finished:
        print("    no finished run directory under %s yet" % root, flush=True)
    for directory in finished:
        scored = directory / "scored_examples.jsonl"
        n_scored = len(scored.read_text(encoding="utf-8").splitlines()) if scored.exists() else 0
        print("    run %s (%s scored examples)" % (directory, n_scored), flush=True)
    return finished


def mirror_outputs(root=None):
    """Copy finished run artifacts (or one file) to Drive so a killed session keeps its evidence."""
    source = Path(root) if root is not None else OUTPUT_ROOT
    if not source.exists():
        print("    %s does not exist yet; nothing to mirror." % source, flush=True)
        return None
    if not Path("/content/drive/MyDrive").exists():
        print("    Drive is not mounted (run the Drive mount cell above); %s stays in the ephemeral Colab VM." % source, flush=True)
        print("    Export it with the final export cell before the session ends, or the results are lost.", flush=True)
        return None
    if source.is_dir():
        shutil.copytree(source, DRIVE_MIRROR, dirs_exist_ok=True)
        print("    mirrored %s -> %s" % (source, DRIVE_MIRROR), flush=True)
        return DRIVE_MIRROR
    destination = DRIVE_MIRROR.parent / source.name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    print("    mirrored %s -> %s" % (source, destination), flush=True)
    return destination

def run_dirs(root):
    return sorted({p.parent for p in Path(root).rglob("manifest.json") if p.is_file()})

def validate_real_run(directory):
    directory = Path(directory)
    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))
    evidence = manifest.get("evidence") or {}
    if evidence.get("mode") != "LOCAL_REAL_MODEL" or evidence.get("real_model_execution") is not True:
        raise RuntimeError(f"Unexpected evidence metadata in {directory}: {evidence}")
    if not (manifest.get("model") or {}).get("model_id"):
        raise RuntimeError(f"Model identity missing from {directory}")
    adapter = ((manifest.get("extra") or {}).get("adapter") or {})
    profile = adapter.get("runtime_profile") or manifest.get("runtime_profile") or (manifest.get("environment") or {}).get("runtime_profile")
    if not profile:
        raise RuntimeError(f"Runtime profile missing from {directory}")
    for name in ("report.md", "report.html", "metrics.json", "config.resolved.yaml", "confidence_intervals.json", "failures.jsonl", "failure_fingerprint.json", "gate_report.json"):
        if not (directory / name).exists():
            raise RuntimeError(f"Required artifact missing: {directory / name}")
    return directory

write_private_config()

PosixPath('configs/studies/phase8_sarvam_application_study/local_user_config.yaml')

## 0. Authenticate for gated models (only for gated repositories)

Most models in this study are public and need nothing here. Gated repositories — Google's Gemma
models, Meta's Llama models — additionally require a Hugging Face account that has **accepted the
model licence**, plus a token. A download that fails with `GatedRepoError` or
`401 Unauthorized` is an authentication problem, not a memory or configuration problem.

1. Sign in to the model page on huggingface.co (for example the Gemma or Llama repository you selected) with the account that
   will run the notebook and accept the licence. Acceptance is per account.
2. Create a read-only token at <https://huggingface.co/settings/tokens>.
3. In Colab, add it as a secret: the key icon in the left panel, new secret, key `HF_TOKEN`. Never
   paste a token into a notebook cell, a config file, or a commit.

The cell below authenticates from that secret, verifies the model is actually reachable, and prints
the commit SHA to use as `MODEL_REVISION`. The token is also exported into the environment, so the
`platform-*` subprocesses that actually download the weights can read it.


In [5]:
# pyright: reportUndefinedVariable=false
import os
from huggingface_hub import get_token, login, model_info

GATED_HINT = (
    "This model is gated. (1) Accept its licence at https://huggingface.co/{model} while signed in as "
    "the account that runs this notebook. (2) Create a read-only token at "
    "https://huggingface.co/settings/tokens. (3) In Colab add it as the secret HF_TOKEN (key icon in "
    "the left panel). Never put a token in a notebook, a config, or a commit. Public models need none "
    "of this."
)


def authenticate_hugging_face():
    """Use a token only if this model is gated; never print or store it in the notebook."""
    if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or get_token():
        print("Hugging Face: a token is already available to this session.")
        return
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = None
    if not token:
        print("Hugging Face: no token found. Public models download anonymously; gated ones need HF_TOKEN.")
        return
    os.environ["HF_TOKEN"] = token
    login(token=token, add_to_git_credential=False)
    print("Hugging Face: authenticated from the Colab secret (token is not shown).")


def check_model_access(model_id=None):
    """Fail fast with instructions when the Hub refuses the model, and show the SHA to pin."""
    model_id = model_id or MODEL_ID
    authenticate_hugging_face()
    try:
        info = model_info(model_id)
    except Exception as exc:
        raise RuntimeError(GATED_HINT.format(model=model_id) + f"\n\nUnderlying error: {exc}") from exc
    print("model access OK:", model_id)
    print("pin this revision for reproducibility: MODEL_REVISION =", info.sha)
    return info.sha


PINNED_REVISION = check_model_access()


Hugging Face: a token is already available to this session.
model access OK: meta-llama/Llama-3.2-3B-Instruct
pin this revision for reproducibility: MODEL_REVISION = 0cb88a4f764b7a12671c53f0838cd831a0843b95


## 2. Mount Google Drive so results survive the session

The Colab VM is wiped when the session ends, and Colab only saves outputs for cells that finished. Mounting Drive lets every completed step copy its run directory out of the VM immediately, and it is what makes the final export cell able to keep your results. Approve the authorization prompt when it appears. This cell mounts a filesystem; it does not download model weights. If you skip it the workflow still runs, but nothing is copied out and you must export before the session dies.

In [6]:
# pyright: reportUndefinedVariable=false
DRIVE_MOUNT = Path("/content/drive")
try:
    from google.colab import drive
except ImportError:
    print("google.colab is unavailable (not a Colab runtime); runs stay on this machine at %s" % OUTPUT_ROOT.resolve())
else:
    if (DRIVE_MOUNT / "MyDrive").exists():
        print("Drive already mounted at", DRIVE_MOUNT)
    else:
        drive.mount(str(DRIVE_MOUNT))
    print("Drive mounted:", (DRIVE_MOUNT / "MyDrive").exists())
print("run mirror target: %s" % DRIVE_MIRROR)
print("If this printed 'Drive is not mounted' later, the steps below will tell you instead of copying silently.")

Mounted at /content/drive
Drive mounted: True
run mirror target: /content/drive/MyDrive/apertus_runs/phase8_real_colab/Llama-3.2-3B-Instruct
If this printed 'Drive is not mounted' later, the steps below will tell you instead of copying silently.


## 3. Core real-model workflow

Run the Drive mount cell above first: without it, nothing is copied out of the ephemeral VM.

The next five cells run, in order: the local model baseline, the six-condition variance matrix (three seeds by two prompt templates), the English RAG/agent suite, the sanitized safety suite, and the collection of the compatible core runs. The first call downloads model weights from Hugging Face into the session cache, so a step that looks idle at its start is downloading weights and Hugging Face progress bars stream in the output. After that, each step prints the command it runs and a heartbeat while that command is in flight, because a platform run writes its directory only when the whole run finishes and prints nothing per example. Each step then lists its finished runs with scored-example counts and copies them to the mounted Drive. The offline mock smoke already ran in setup. Each step first releases any model still held by this kernel (free_gpu) and prints free GPU memory, because every platform command runs in a fresh process: a model left loaded by the manual GPU test would otherwise make that process run out of memory even when the model itself fits. If a step still fails with a CUDA out-of-memory error, restart the runtime and re-run that cell, lower MAX_NEW_TOKENS, or set QUANTIZATION to int8 for models above about 4B. This workflow will stop rather than use CPU when CUDA is unavailable.

In [7]:
# pyright: reportUndefinedVariable=false
check_model_access()
require_cuda()

# Step 1/4 - real-model baseline (one condition).
run_platform("platform-run", "--config", PRIVATE_CONFIG, "--out", OUTPUT_ROOT / "local_baseline")
baseline_runs = run_dirs(OUTPUT_ROOT / "local_baseline")
if not baseline_runs:
    raise RuntimeError("No real baseline artifact was produced.")
validate_real_run(baseline_runs[-1])
print_artifacts(OUTPUT_ROOT / "local_baseline")
mirror_outputs()

# Step 2/4 - variance matrix. A run writes its directory only when it
# finishes, so use the run_platform heartbeat as the liveness signal instead
# of expecting per-example progress output.
VARIANCE_SEEDS = [11, 22, 33]
VARIANCE_PROMPTS = ["base", "strict"]
# Quick complete pass (two cells, still enough for the primary comparison):
#   VARIANCE_SEEDS = [11, 22]
#   VARIANCE_PROMPTS = ["base"]
# Record any deviation from the pre-registered matrix in the study notes.
variance_config = write_experiment_config("local_variance.yaml", CONFIG_ROOT / "local_variance.yaml", OUTPUT_ROOT / "local_variance", seeds=VARIANCE_SEEDS, prompt_templates=VARIANCE_PROMPTS)
run_platform("platform-matrix", "--config", variance_config, "--out", OUTPUT_ROOT / "local_variance")
print_artifacts(OUTPUT_ROOT / "local_variance")
mirror_outputs()

Hugging Face: a token is already available to this session.
model access OK: meta-llama/Llama-3.2-3B-Instruct
pin this revision for reproducibility: MODEL_REVISION = 0cb88a4f764b7a12671c53f0838cd831a0843b95
{
  "schema_version": "1.0",
  "runtime_environment": "google_colab",
  "os": {
    "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
    "system": "Linux",
    "release": "6.6.122+",
    "machine": "x86_64",
    "processor": "x86_64",
    "cpu_count": 2
  },
  "python": {
    "version": "3.13.15",
    "implementation": "CPython",
    "executable": "/usr/bin/python3"
  },
  "torch_version": "2.11.0+cu128",
  "cuda_version": "12.8",
  "gpu_name": "Tesla T4",
  "gpu_count": 1,
  "gpu_memory": 14.56,
  "gpu_memory_unit": "GiB",
  "cpu_count": 2,
  "device_selection": "cuda",
  "precision": "float16",
  "quantization": "none",
  "hardware_measured": true,
  "hardware_measurement_status": "measured",
  "latency_method": "client_side_wall_clock",
  "model_generation_latency": null,
  "e

PosixPath('/content/drive/MyDrive/apertus_runs/phase8_real_colab/Llama-3.2-3B-Instruct')

In [8]:
# pyright: reportUndefinedVariable=false
# Step 3/4 - English RAG and tool-agent episode matrix (two cells).
rag_config = write_experiment_config("local_rag_agent.yaml", CONFIG_ROOT / "local_rag_agent.yaml", OUTPUT_ROOT / "local_rag_agent")
run_platform("platform-matrix", "--config", rag_config, "--out", OUTPUT_ROOT / "local_rag_agent")
print_artifacts(OUTPUT_ROOT / "local_rag_agent")
mirror_outputs()

    GPU before step: {'cuda': True, 'gpu': 'Tesla T4', 'free_gib': 14.46, 'total_gib': 14.56, 'allocated_gib': 0.0, 'reserved_gib': 0.0}
$ /usr/bin/python3 -u -m apertus_eval_prep platform-matrix --config runs/phase8_real_colab/configs/local_rag_agent.yaml --out runs/phase8_real_colab/local_rag_agent
    done platform-matrix in 0.2 min
    run runs/phase8_real_colab/local_rag_agent/20260926T171123967800Z-sarvam-application-local-rag-agent-basel-9850359a (5 scored examples)
    run runs/phase8_real_colab/local_rag_agent/20260926T171132697308Z-sarvam-application-local-rag-agent-backe-30b7f819 (5 scored examples)
    mirrored runs/phase8_real_colab -> /content/drive/MyDrive/apertus_runs/phase8_real_colab/Llama-3.2-3B-Instruct


PosixPath('/content/drive/MyDrive/apertus_runs/phase8_real_colab/Llama-3.2-3B-Instruct')

In [9]:
# pyright: reportUndefinedVariable=false
# Step 4/4 - sanitized safety suite.
safety_config = write_experiment_config("local_safety.yaml", CONFIG_ROOT / "local_safety.yaml", OUTPUT_ROOT / "local_safety")
run_platform("platform-safety", "--config", safety_config, "--out", OUTPUT_ROOT / "local_safety")
print_artifacts(OUTPUT_ROOT / "local_safety")
mirror_outputs()

    GPU before step: {'cuda': True, 'gpu': 'Tesla T4', 'free_gib': 14.46, 'total_gib': 14.56, 'allocated_gib': 0.0, 'reserved_gib': 0.0}
$ /usr/bin/python3 -u -m apertus_eval_prep platform-safety --config runs/phase8_real_colab/configs/local_safety.yaml --out runs/phase8_real_colab/local_safety
    done platform-safety in 0.2 min
    run runs/phase8_real_colab/local_safety/20260926T171136151336Z-sarvam-application-local-safety-eb8b5c60 (11 scored examples)
    mirrored runs/phase8_real_colab -> /content/drive/MyDrive/apertus_runs/phase8_real_colab/Llama-3.2-3B-Instruct


PosixPath('/content/drive/MyDrive/apertus_runs/phase8_real_colab/Llama-3.2-3B-Instruct')

In [10]:
# pyright: reportUndefinedVariable=false
# Collect the compatible core runs that the primary comparison may use.
CORE_RUNS = []
for directory in run_dirs(OUTPUT_ROOT / "local_variance"):
    manifest = json.loads((directory / "manifest.json").read_text(encoding="utf-8"))
    if (manifest.get("conditions") or {}).get("prompt_template") == "base":
        validate_real_run(directory)
        CORE_RUNS.append(directory)
CORE_RUNS = sorted(set(CORE_RUNS))
if len(CORE_RUNS) < 2:
    raise RuntimeError("At least two compatible base-template real core runs are required for selection. A single-seed variance matrix cannot satisfy this; add a second base-template seed and re-run the variance cell.")
print("compatible real core runs:", [str(p) for p in CORE_RUNS])
print_artifacts(OUTPUT_ROOT)

compatible real core runs: ['runs/phase8_real_colab/local_variance/20260926T171038021413Z-sarvam-application-local-variance-baseli-a8270a86', 'runs/phase8_real_colab/local_variance/20260926T171050711386Z-sarvam-application-local-variance-backen-365a22bb', 'runs/phase8_real_colab/local_variance/20260926T171056850859Z-sarvam-application-local-variance-backen-eb9d0318']
    run runs/phase8_real_colab/local_baseline/20260926T170957194115Z-sarvam-application-local-user-baseline-65505f60 (4 scored examples)
    run runs/phase8_real_colab/local_rag_agent/20260926T171123967800Z-sarvam-application-local-rag-agent-basel-9850359a (5 scored examples)
    run runs/phase8_real_colab/local_rag_agent/20260926T171132697308Z-sarvam-application-local-rag-agent-backe-30b7f819 (5 scored examples)
    run runs/phase8_real_colab/local_safety/20260926T171136151336Z-sarvam-application-local-safety-eb8b5c60 (11 scored examples)
    run runs/phase8_real_colab/local_variance/20260926T171038021413Z-sarvam-applicat

[PosixPath('runs/phase8_real_colab/local_baseline/20260926T170957194115Z-sarvam-application-local-user-baseline-65505f60'),
 PosixPath('runs/phase8_real_colab/local_rag_agent/20260926T171123967800Z-sarvam-application-local-rag-agent-basel-9850359a'),
 PosixPath('runs/phase8_real_colab/local_rag_agent/20260926T171132697308Z-sarvam-application-local-rag-agent-backe-30b7f819'),
 PosixPath('runs/phase8_real_colab/local_safety/20260926T171136151336Z-sarvam-application-local-safety-eb8b5c60'),
 PosixPath('runs/phase8_real_colab/local_variance/20260926T171038021413Z-sarvam-application-local-variance-baseli-a8270a86'),
 PosixPath('runs/phase8_real_colab/local_variance/20260926T171050711386Z-sarvam-application-local-variance-backen-365a22bb'),
 PosixPath('runs/phase8_real_colab/local_variance/20260926T171056850859Z-sarvam-application-local-variance-backen-eb9d0318'),
 PosixPath('runs/phase8_real_colab/local_variance/20260926T171103123073Z-sarvam-application-local-variance-backen-36885d42'),
 Po

In [11]:
from pathlib import Path
import os
import subprocess

cache_root = Path.home() / ".cache" / "huggingface" / "hub"

print("Hugging Face cache root:", cache_root)
print("Cache exists:", cache_root.exists())

if cache_root.exists():
    result = subprocess.run(
        ["du", "-sh", str(cache_root)],
        capture_output=True,
        text=True,
    )
    print("Cache size:", result.stdout.strip() or "unknown")

    print("\nLlama-related cache directories:")
    model_dirs = sorted(
        [path for path in cache_root.glob("*Llama*") if path.is_dir()]
    )

    if not model_dirs:
        print("No matching model cache directory found.")
    else:
        for directory in model_dirs:
            size = subprocess.run(
                ["du", "-sh", str(directory)],
                capture_output=True,
                text=True,
            ).stdout.strip()
            print("-", directory.name, "|", size)

            incomplete = list(directory.rglob("*.incomplete"))
            if incomplete:
                print("  Incomplete download files:", len(incomplete))
else:
    print("No Hugging Face model cache found.")

Hugging Face cache root: /root/.cache/huggingface/hub
Cache exists: False
No Hugging Face model cache found.


In [12]:
import gc
import time
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Restart the Colab runtime with GPU enabled.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory before load:",
    round(torch.cuda.memory_allocated() / 1024**3, 3),
    "GiB",
)

# A gated repository needs a licence acceptance plus a token. The section 0 preflight
# turns a Hub 401 from this cell into one clear message instead of a deep traceback.
if "check_model_access" in globals():
    check_model_access()
else:
    print("note: run the section 0 authentication cell above for a clear gated-model message.")
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
MODEL_REVISION = "0cb88a4f764b7a12671c53f0838cd831a0843b95"

print("\n[1/4] Loading tokenizer from cache...")
start = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=False,
)

print(
    "Tokenizer loaded in",
    round(time.perf_counter() - start, 2),
    "seconds",
)

print("\n[2/4] Loading model weights in float16...")
start = time.perf_counter()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=False,
)

print(
    "Model weights loaded in",
    round(time.perf_counter() - start, 2),
    "seconds",
)

print("\n[3/4] Moving model to Tesla T4 GPU...")
start = time.perf_counter()

model = model.to("cuda")
model.eval()

print(
    "Model moved to GPU in",
    round(time.perf_counter() - start, 2),
    "seconds",
)

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 3),
    "GiB",
)

print(
    "GPU memory reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 3),
    "GiB",
)

print("\n[4/4] Running one short deterministic generation...")

prompt = "Reply with exactly one sentence: What is an evaluation metric?"

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to("cuda")

start = time.perf_counter()

with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=32,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

latency_seconds = time.perf_counter() - start

response = tokenizer.decode(
    generated[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("\nGeneration latency:", round(latency_seconds, 3), "seconds")
print("Generated response:")
print(response)

print("\nDIRECT GPU MODEL TEST PASSED")

del generated
del inputs
del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

print("GPU cache cleared.")

Torch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory before load: 0.0 GiB
Hugging Face: a token is already available to this session.
model access OK: meta-llama/Llama-3.2-3B-Instruct
pin this revision for reproducibility: MODEL_REVISION = 0cb88a4f764b7a12671c53f0838cd831a0843b95

[1/4] Loading tokenizer from cache...


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model weights loaded in 124.46 seconds

[3/4] Moving model to Tesla T4 GPU...
Model moved to GPU in 1.89 seconds
GPU memory allocated: 5.985 GiB
GPU memory reserved: 6.029 GiB

[4/4] Running one short deterministic generation...


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Generation latency: 2.577 seconds
Generated response:
 An evaluation metric is a quantifiable measure used to assess the performance of a system, model, or algorithm, providing a way to compare its effectiveness and accuracy.

DIRECT GPU MODEL TEST PASSED
GPU cache cleared.


## 4. Optional India-context diagnostic

Run only after core English results are complete. This is an exploratory diagnostic, not a multilingual benchmark, and it must not replace the English-first primary results.

In [13]:
# pyright: reportUndefinedVariable=false
RUN_INDIA_CONTEXT_DIAGNOSTIC = False
if RUN_INDIA_CONTEXT_DIAGNOSTIC:
    require_cuda()
    india_config = write_experiment_config("india_context_diagnostic.yaml", CONFIG_ROOT / "india_context_diagnostic.yaml", OUTPUT_ROOT / "india_context_diagnostic")
    run_platform("platform-run", "--config", india_config, "--out", OUTPUT_ROOT / "india_context_diagnostic")
else:
    print("Skipped optional India-context diagnostic; core English results are the primary study.")

Skipped optional India-context diagnostic; core English results are the primary study.


## 5. Ingest, select, report, review, and analyze

Only real LOCAL_REAL_MODEL core runs are passed to the primary deployment comparison. Mock smoke artifacts remain separate under mock_smoke. Do not use --allow-incompatible. Section 6 then exports and downloads everything this section produces.

In [14]:
# pyright: reportUndefinedVariable=false
import json
from pathlib import Path
from apertus_eval_prep.utils.serialization import read_yaml, write_yaml

points_path = OUTPUT_ROOT / "comparison_points.json"
selection_path = OUTPUT_ROOT / "selection.json"
run_platform("platform-ingest-runs", "--runs", *map(str, CORE_RUNS), "--out", points_path)
run_platform("platform-select", "--points", points_path, "--constraints", STUDY_DIR / "selection_constraints.json", "--out", selection_path)
report_run = CORE_RUNS[0]
run_platform("platform-report", "--run", report_run, "--format", "both", "--out", report_run)
run_platform("platform-fingerprint", "--run", report_run)
review_package = OUTPUT_ROOT / "review_package.jsonl"
run_platform("platform-export-review", "--run", report_run, "--dimensions", "correctness", "groundedness", "safe_behavior", "instruction_following", "tool_use_correctness", "--sample-size", "30", "--sampling-strategy", "stratified", "--seed", "7", "--study-id", "sarvam_application_real_eval_v1", "--out", review_package)
print("Review package written. Templates and empty packages are not human evidence.")
completed_annotations = Path("YOUR_COMPLETED_ANNOTATIONS.jsonl")
review_results = OUTPUT_ROOT / "review_results.json"
if completed_annotations.exists():
    run_platform("platform-ingest-review", "--input", completed_annotations, "--study-id", "sarvam_application_real_eval_v1", "--out", review_results)
study_config = read_yaml(STUDY_DIR / "study_metadata.yaml")
study_config.setdefault("study", {})["matrix"] = {"models": [MODEL_ID], "prompt_templates": ["base", "strict"], "seeds": [11, 22, 33], "dtype": [DTYPE], "quantization": [QUANTIZATION], "decoding": {"temperature": [0.0], "top_p": [1.0]}}
study_config["base"] = _set_identity({"base": study_config.get("base", {})}, OUTPUT_ROOT / "study")["base"]
private_study = CONFIG_ROOT / "study_metadata.yaml"
write_yaml(private_study, study_config)
study_args = ["platform-study-analyze", "--study-config", private_study, "--runs", *map(str, CORE_RUNS), "--out", OUTPUT_ROOT / "study"]
if review_results.exists():
    study_args += ["--reviews", review_results]
run_platform(*study_args)
print("Section 6 exports and downloads the artifacts; do not commit caches, credentials, private data, or raw private outputs.")

    GPU before step: {'cuda': True, 'gpu': 'Tesla T4', 'free_gib': 14.41, 'total_gib': 14.56, 'allocated_gib': 0.008, 'reserved_gib': 0.02}
$ /usr/bin/python3 -u -m apertus_eval_prep platform-ingest-runs --runs runs/phase8_real_colab/local_variance/20260926T171038021413Z-sarvam-application-local-variance-baseli-a8270a86 runs/phase8_real_colab/local_variance/20260926T171050711386Z-sarvam-application-local-variance-backen-365a22bb runs/phase8_real_colab/local_variance/20260926T171056850859Z-sarvam-application-local-variance-backen-eb9d0318 --out runs/phase8_real_colab/comparison_points.json
    done platform-ingest-runs in 0.0 min
    GPU before step: {'cuda': True, 'gpu': 'Tesla T4', 'free_gib': 14.41, 'total_gib': 14.56, 'allocated_gib': 0.008, 'reserved_gib': 0.02}
$ /usr/bin/python3 -u -m apertus_eval_prep platform-select --points runs/phase8_real_colab/comparison_points.json --constraints configs/studies/phase8_sarvam_application_study/selection_constraints.json --out runs/phase8_re

## 6. Export and download the results

Run this after the analysis cell. It writes one zip of everything under `runs/phase8_real_colab`, copies the zip to the mounted Drive, and starts a browser download through `google.colab.files`. Nothing downloads on its own: Colab keeps results in the VM until a cell saves or exports them, and the VM is deleted when the session ends. If the browser download does not start, open the Colab **Files** pane and download the archive from there, or take it from Drive. Do not commit model caches, credentials, private data, or raw private outputs.

In [15]:
# pyright: reportUndefinedVariable=false
# Build one archive of every finished run, mirror it to Drive, then download it.
if not OUTPUT_ROOT.exists():
    raise RuntimeError("Nothing to export: %s does not exist. Run the workflow cells first." % OUTPUT_ROOT)
export_dir = Path("/content") if Path("/content").is_dir() else Path.cwd()
archive = Path(shutil.make_archive(str(export_dir / "apertus_phase8_real_colab_Llama-3.2-3B-Instruct_export"), "zip", root_dir=str(OUTPUT_ROOT)))
finished = run_dirs(OUTPUT_ROOT)
print("archived %d finished run(s) from %s" % (len(finished), OUTPUT_ROOT))
for directory in finished:
    print("  %s" % directory)
print("archive: %s (%.1f MB)" % (archive, archive.stat().st_size / 1_000_000.0))
mirror_outputs(archive)
try:
    from google.colab import files
except ImportError:
    print("Not a Colab runtime; the archive is on this machine at %s" % archive)
else:
    try:
        files.download(str(archive))
        print("Browser download started. If it does not appear, use the Colab Files pane or the Drive copy.")
    except Exception as exc:  # noqa: BLE001 - Colab raises several unrelated download errors
        print("Browser download failed (%s); use the Colab Files pane or the Drive copy at %s" % (exc, DRIVE_MIRROR.parent))
print("Save this notebook (File > Save) only after this cell finishes, so the saved outputs include the results.")

archived 11 finished run(s) from runs/phase8_real_colab
  runs/phase8_real_colab/local_baseline/20260926T170957194115Z-sarvam-application-local-user-baseline-65505f60
  runs/phase8_real_colab/local_rag_agent/20260926T171123967800Z-sarvam-application-local-rag-agent-basel-9850359a
  runs/phase8_real_colab/local_rag_agent/20260926T171132697308Z-sarvam-application-local-rag-agent-backe-30b7f819
  runs/phase8_real_colab/local_safety/20260926T171136151336Z-sarvam-application-local-safety-eb8b5c60
  runs/phase8_real_colab/local_variance/20260926T171038021413Z-sarvam-application-local-variance-baseli-a8270a86
  runs/phase8_real_colab/local_variance/20260926T171050711386Z-sarvam-application-local-variance-backen-365a22bb
  runs/phase8_real_colab/local_variance/20260926T171056850859Z-sarvam-application-local-variance-backen-eb9d0318
  runs/phase8_real_colab/local_variance/20260926T171103123073Z-sarvam-application-local-variance-backen-36885d42
  runs/phase8_real_colab/local_variance/20260926T17

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Browser download started. If it does not appear, use the Colab Files pane or the Drive copy.
Save this notebook (File > Save) only after this cell finishes, so the saved outputs include the results.
